# 🛡️ ToxicGuard V5.1 — Sarkazm Destekli Genişletilmiş Veri Seti

> **V5 → V5.1 Farkı:** OffensEval-TR `trust_remote_code` hatası düzeltildi.
> Alternatif Türkçe dataset eklendi. Önceki eğitimden gelen false positive'ler için
> daha fazla sarkazm verisi eklendi.

## 📋 V5 Eğitim Sonuçları (Referans)
| Metrik | V3 Baseline | V5 (önceki) | Hedef V5.1 |
|--------|-------------|-------------|------------|
| F1-macro | 0.693 | **0.6967** ✅ | 0.720+ |
| ROC-AUC | 0.978 | **0.9818** ✅ | 0.985+ |
| Veri boyutu | ~45K | 130K | **~200K** |

## 🐛 V5'te Tespit Edilen Sorunlar
```
❌ 'You're killing it, congrats!'  → Toxic 0.53  (FALSE POSITIVE - iyi niyet)
❌ 'Oh harika fikir, insanları öldürmek tam çözüm' → TEMİZ 0.07  (Türkçe sarkazm kaçırıldı)
❌ OffensEval-TR yüklenemedi       → trust_remote_code desteği kaldırıldı
```

## 📥 Drive'a Yüklenmesi Gereken Dosyalar
**Kesin yol:** `MyDrive/ToxicGuard/data/`

| Dosya | İndir | Not |
|-------|-------|-----|
| `train.csv` | Kaggle Jigsaw | Zaten mevcut ✅ |
| `jigsaw_bias_train.csv` | [Kaggle Jigsaw Bias](https://kaggle.com/c/jigsaw-unintended-bias-in-toxicity-classification) → train.csv → ismini değiştir | Önemli |
| `train-balanced-sarcasm.csv` | [Kaggle SARC](https://kaggle.com/datasets/danofer/sarcasm) | Önemli |

## ⚡ Google Colab Notları
- **Runtime → T4 GPU** seçmeyi unutma!
- Tahmini süre: **~30-45 dk** (T4 GPU, 2 epoch, ~200K veri)


---
## 🔧 BÖLÜM 1 — Kurulum & Drive Bağlantısı

In [ ]:
# HÜCRE 1: Gerekli Paketlerin Kurulumu
!pip install transformers datasets evaluate accelerate scikit-learn pandas numpy --quiet
print("✅ Tüm paketler kuruldu!")

In [ ]:
# HÜCRE 2: Google Drive Bağlantısı
import os
from google.colab import drive
drive.mount('/content/drive')

BASE        = '/content/drive/MyDrive/ToxicGuard'
MODELS_DIR  = os.path.join(BASE, 'models')
DATA_DIR    = os.path.join(BASE, 'data')
RESULTS_DIR = os.path.join(BASE, 'reports', 'model_results')

for d in [MODELS_DIR, DATA_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Klasör yolları hazır!")
print(f"  DATA_DIR    → {DATA_DIR}")
print()

# Drive'daki data/ klasörünü listele — hangi dosyaların mevcut olduğunu göster
print("📂 Drive'daki data/ klasörü içeriği:")
if os.path.exists(DATA_DIR):
    for f in sorted(os.listdir(DATA_DIR)):
        size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / (1024*1024)
        print(f"   {'✅' if f.endswith('.csv') else '📁'} {f}  ({size_mb:.1f} MB)")
else:
    print("   (klasör boş veya bulunamadı)")

In [ ]:
# HÜCRE 3: Kütüphaneleri Dahil Etme
import pandas as pd
import numpy as np
import torch
import json
import warnings
warnings.filterwarnings('ignore')

from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction
)
from sklearn.metrics import f1_score, roc_auc_score

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
MODEL_NAME = 'xlm-roberta-base'

device_name = 'GPU Aktif! 🚀' if torch.cuda.is_available() else 'CPU ⚠️ — Runtime → T4 GPU seç!'
print(f"Cihaz: {device_name}")
print(f"Etiketler: {LABEL_COLS}")

---
## 📂 BÖLÜM 2 — Genişletilmiş Veri Seti (~200K)

```
┌───────────────────────────────────┬────────┬──────────────────────────────┐
│ Kaynak                            │ Boyut  │ Yükleme Yöntemi              │
├───────────────────────────────────┼────────┼──────────────────────────────┤
│ Kaggle Jigsaw (orijinal)          │ ~48K   │ Drive CSV (mevcut) ✅        │
│ Jigsaw Unintended Bias            │ ~50K   │ Drive CSV (manuel yükle) ⬇️  │
│ SemEval-2018 Irony (EN)           │ ~3.8K  │ wget (otomatik) ✅           │
│ SARC Reddit (dengeli, EN)         │ ~30K   │ Drive CSV (manuel yükle) ⬇️  │
│ Overfit-GM Turkish                │ ~77K   │ HuggingFace (otomatik) ✅    │
│ Toygar Türkçe Offensive [YENİ]   │ ~5K    │ HuggingFace (otomatik) ✅    │
└───────────────────────────────────┴────────┴──────────────────────────────┘
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# HÜCRE 4 — VERİ 1: Orijinal Kaggle Jigsaw (~48K dengeli örneklem)
# Drive yolu: MyDrive/ToxicGuard/data/train.csv  ← zaten mevcut!
# ─────────────────────────────────────────────────────────────────────
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')

df_orig = pd.read_csv(TRAIN_CSV)

toxic_mask = df_orig[LABEL_COLS].sum(axis=1) > 0
df_toxic   = df_orig[toxic_mask]
df_safe    = df_orig[~toxic_mask].sample(n=min(len(df_toxic) * 2, (~toxic_mask).sum()), random_state=42)
df_en      = pd.concat([df_toxic, df_safe]).sample(frac=1, random_state=42).reset_index(drop=True)
df_en      = df_en[['comment_text'] + LABEL_COLS]

print(f"✅ Kaggle Orijinal → {df_en.shape[0]:,} satır")
print(f"   Toksik: {len(df_toxic):,} | Zararsız örneklem: {len(df_safe):,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# HÜCRE 5 — VERİ 2: Jigsaw Unintended Bias (~50K örtük toksisite)
#
# Eksikse şu adımları yap:
#   1. https://www.kaggle.com/c/jigsaw-unintended-bias-in-toxicity-classification
#   2. Data sekmesi → train.csv'yi indir
#   3. İndirilen dosyanın adını 'jigsaw_bias_train.csv' olarak değiştir
#   4. Google Drive → MyDrive → ToxicGuard → data → klasörüne sürükle
# ─────────────────────────────────────────────────────────────────────
JIGSAW_BIAS_CSV = os.path.join(DATA_DIR, 'jigsaw_bias_train.csv')

if os.path.exists(JIGSAW_BIAS_CSV):
    df_bias_raw = pd.read_csv(JIGSAW_BIAS_CSV)

    df_jigsaw = pd.DataFrame()
    df_jigsaw['comment_text']  = df_bias_raw['comment_text']
    df_jigsaw['toxic']         = (df_bias_raw['toxicity']        >= 0.5).astype(int)
    df_jigsaw['severe_toxic']  = (df_bias_raw['severe_toxicity'] >= 0.5).astype(int)
    df_jigsaw['obscene']       = (df_bias_raw.get('obscene',  pd.Series([0]*len(df_bias_raw))) >= 0.5).astype(int)
    df_jigsaw['threat']        = (df_bias_raw.get('threat',   pd.Series([0]*len(df_bias_raw))) >= 0.5).astype(int)
    df_jigsaw['insult']        = (df_bias_raw.get('insult',   pd.Series([0]*len(df_bias_raw))) >= 0.5).astype(int)
    df_jigsaw['identity_hate'] = (df_bias_raw.get('identity_attack', pd.Series([0]*len(df_bias_raw))) >= 0.5).astype(int)
    df_jigsaw = df_jigsaw.dropna(subset=['comment_text']).reset_index(drop=True)

    BIAS_SAMPLE = 50_000
    toxic_bias  = df_jigsaw[df_jigsaw['toxic'] == 1]
    safe_bias   = df_jigsaw[df_jigsaw['toxic'] == 0]
    safe_bias   = safe_bias.sample(n=min(BIAS_SAMPLE - len(toxic_bias), len(safe_bias)), random_state=42)
    df_jigsaw   = pd.concat([toxic_bias, safe_bias]).sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"✅ Jigsaw Unintended Bias → {df_jigsaw.shape[0]:,} satır")
    print(f"   Toksik: {df_jigsaw['toxic'].sum():,} | Zararsız: {(df_jigsaw['toxic']==0).sum():,}")
else:
    print("⏭️  jigsaw_bias_train.csv bulunamadı → atlanıyor.")
    print(f"   Beklenen yol: {JIGSAW_BIAS_CSV}")
    df_jigsaw = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# HÜCRE 6 — VERİ 3: SemEval-2018 Task 3 (~3.8K) — GitHub'dan otomatik
# ─────────────────────────────────────────────────────────────────────
SEMEVAL_DIR  = '/content/semeval2018_task3'
os.makedirs(SEMEVAL_DIR, exist_ok=True)
SEMEVAL_URL  = ('https://raw.githubusercontent.com/Cyvhee/SemEval2018-Task3/master/'
                'datasets/train/SemEval2018-T3-train-taskA_emoji.txt')
SEMEVAL_FILE = f'{SEMEVAL_DIR}/semeval_train.txt'

!wget -q -O {SEMEVAL_FILE} {SEMEVAL_URL}

if os.path.exists(SEMEVAL_FILE) and os.path.getsize(SEMEVAL_FILE) > 200:
    rows = []
    with open(SEMEVAL_FILE, 'r', encoding='utf-8') as f:
        next(f)
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                rows.append({'text': parts[2], 'is_ironic': int(parts[1])})

    df_semeval_raw = pd.DataFrame(rows)
    df_semeval = pd.DataFrame()
    df_semeval['comment_text']  = df_semeval_raw['text']
    df_semeval['toxic']         = df_semeval_raw['is_ironic'].astype(int)
    df_semeval['severe_toxic']  = 0
    df_semeval['obscene']       = 0
    df_semeval['threat']        = 0
    df_semeval['insult']        = df_semeval_raw['is_ironic'].astype(int)
    df_semeval['identity_hate'] = 0

    print(f"✅ SemEval-2018 Irony → {df_semeval.shape[0]:,} tweet")
    print(f"   Alaycı: {df_semeval['toxic'].sum():,} | Normal: {(df_semeval['toxic']==0).sum():,}")
else:
    print("⚠️  SemEval indirilemedi → atlanıyor.")
    df_semeval = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# HÜCRE 7 — VERİ 4: SARC Reddit Corpus Dengeli (~30K)
#
# Eksikse şu adımları yap:
#   1. https://www.kaggle.com/datasets/danofer/sarcasm
#   2. 'train-balanced-sarcasm.csv' dosyasını indir
#   3. Google Drive → MyDrive → ToxicGuard → data → klasörüne sürükle
#
# Sütunlar: 'label' (1=sarkastik), 'comment' (metin)
# ─────────────────────────────────────────────────────────────────────
SARC_CSV = os.path.join(DATA_DIR, 'train-balanced-sarcasm.csv')

if os.path.exists(SARC_CSV):
    df_sarc_raw = pd.read_csv(SARC_CSV)

    SARC_SAMPLE = 30_000
    if len(df_sarc_raw) > SARC_SAMPLE:
        df_sarc_raw = df_sarc_raw.sample(n=SARC_SAMPLE, random_state=42).reset_index(drop=True)

    text_col = 'comment' if 'comment' in df_sarc_raw.columns else df_sarc_raw.columns[0]
    df_sarc = pd.DataFrame()
    df_sarc['comment_text']  = df_sarc_raw[text_col].astype(str)
    df_sarc['toxic']         = df_sarc_raw['label'].astype(int)
    df_sarc['severe_toxic']  = 0
    df_sarc['obscene']       = 0
    df_sarc['threat']        = 0
    df_sarc['insult']        = 0
    df_sarc['identity_hate'] = 0

    df_sarc = df_sarc[df_sarc['comment_text'].str.len() > 10].reset_index(drop=True)

    print(f"✅ SARC Reddit (dengeli) → {df_sarc.shape[0]:,} yorum")
    print(f"   Sarkastik: {df_sarc['toxic'].sum():,} | Normal: {(df_sarc['toxic']==0).sum():,}")
else:
    print("⏭️  train-balanced-sarcasm.csv bulunamadı → atlanıyor.")
    print(f"   Beklenen yol: {SARC_CSV}")
    df_sarc = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# HÜCRE 8 — VERİ 5: Türkçe Overfit-GM (~77K) [Otomatik]
# ─────────────────────────────────────────────────────────────────────
print("🇹🇷 Overfit-GM Türkçe yükleniyor...")
try:
    tr_dataset = load_dataset("Overfit-GM/turkish-toxic-language", split="train")
    df_tr_raw  = pd.DataFrame(tr_dataset)

    df_tr = pd.DataFrame()
    df_tr['comment_text']  = df_tr_raw['text']
    df_tr['toxic']         = df_tr_raw['is_toxic']
    df_tr['severe_toxic']  = 0
    df_tr['obscene']       = (df_tr_raw['target'] == 'PROFANITY').astype(int)
    df_tr['threat']        = 0
    df_tr['insult']        = (df_tr_raw['target'] == 'INSULT').astype(int)
    df_tr['identity_hate'] = df_tr_raw['target'].isin(['RACIST', 'SEXIST']).astype(int)

    print(f"✅ Overfit-GM Türkçe → {df_tr.shape[0]:,} yorum")
except Exception as e:
    print(f"⚠️  Overfit-GM yüklenemedi: {e}")
    df_tr = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# HÜCRE 9 — VERİ 6: Toygar Türkçe Offensive Language [YENİ - Otomatik]
#
# OffensEval-TR'nin trust_remote_code hatası nedeniyle bu alternatif
# kullanılıyor. Standart HuggingFace formatında, sorunsuz yüklenir.
# ─────────────────────────────────────────────────────────────────────
print("🇹🇷 Toygar Türkçe Offensive Language yükleniyor (HuggingFace)...")
try:
    toygar_ds = load_dataset("Toygar/turkish-offensive-language-detection")

    splits = []
    for split_name in ['train', 'validation', 'test']:
        if split_name in toygar_ds:
            splits.append(pd.DataFrame(toygar_ds[split_name]))
    df_toygar_raw = pd.concat(splits).reset_index(drop=True)

    print(f"  Sütunlar: {list(df_toygar_raw.columns)}")

    # Sütun adını otomatik bul
    text_col  = next((c for c in ['text', 'sentence', 'tweet', 'comment'] if c in df_toygar_raw.columns), df_toygar_raw.columns[0])
    label_col = next((c for c in ['label', 'offensive', 'is_offensive', 'target'] if c in df_toygar_raw.columns), None)

    df_toygar = pd.DataFrame()
    df_toygar['comment_text'] = df_toygar_raw[text_col].astype(str)

    if label_col:
        # Etiket: 1 veya 'offensive' → toxic=1
        df_toygar['toxic'] = df_toygar_raw[label_col].apply(
            lambda x: 1 if str(x) in ['1', 'offensive', 'OFF', 'True', 'true'] else 0
        )
    else:
        df_toygar['toxic'] = 0

    df_toygar['severe_toxic']  = 0
    df_toygar['obscene']       = 0
    df_toygar['threat']        = 0
    df_toygar['insult']        = df_toygar['toxic'].copy()
    df_toygar['identity_hate'] = 0

    df_toygar = df_toygar.dropna(subset=['comment_text']).reset_index(drop=True)
    df_toygar = df_toygar[df_toygar['comment_text'].str.len() > 5]

    print(f"✅ Toygar TR Offensive → {df_toygar.shape[0]:,} yorum")
    print(f"   Offensive: {df_toygar['toxic'].sum():,} | Not: {(df_toygar['toxic']==0).sum():,}")

except Exception as e:
    print(f"⚠️  Toygar yüklenemedi: {e}")
    df_toygar = pd.DataFrame(columns=['comment_text'] + LABEL_COLS)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# HÜCRE 10 — TÜM VERİ SETLERİNİ BİRLEŞTİR
# ─────────────────────────────────────────────────────────────────────
print("🔀 Tüm veri setleri birleştiriliyor...")
print("=" * 58)

datasets_info = [
    (df_en,      'Kaggle Jigsaw Orijinal  '),
    (df_jigsaw,  'Jigsaw Unintended Bias  '),
    (df_semeval, 'SemEval-2018 Irony      '),
    (df_sarc,    'SARC Reddit             '),
    (df_tr,      'Overfit-GM Türkçe       '),
    (df_toygar,  'Toygar TR Offensive     '),
]

datasets_list = []
for df_part, name in datasets_info:
    if len(df_part) > 0:
        df_part = df_part[['comment_text'] + LABEL_COLS].copy()
        df_part[LABEL_COLS] = df_part[LABEL_COLS].fillna(0).astype(int)
        df_part = df_part.dropna(subset=['comment_text'])
        df_part = df_part[df_part['comment_text'].str.len() > 5]
        datasets_list.append(df_part)
        print(f"  ✅ {name}: {len(df_part):>8,} satır")
    else:
        print(f"  ⏭️  {name}: atlandı")

print("=" * 58)
df_mixed = pd.concat(datasets_list, ignore_index=True)
df_mixed = df_mixed.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n🌍 TOPLAM KARMA VERİ SETİ: {df_mixed.shape[0]:,} satır")
print(f"\n📊 Etiket Dağılımı:")
for col in LABEL_COLS:
    count = int(df_mixed[col].sum())
    pct   = count / len(df_mixed) * 100
    bar   = '█' * int(pct / 2)
    print(f"   {col:<16}: {count:>7,}  ({pct:4.1f}%) {bar}")

---
## 🤖 BÖLÜM 3 — XLM-RoBERTa Hazırlığı & Tokenization

In [ ]:
# HÜCRE 11: HuggingFace Dataset Formatına Çevirme
labels = df_mixed[LABEL_COLS].values.astype(float)
texts  = df_mixed['comment_text'].tolist()

hf_dataset = Dataset.from_dict({'text': texts, 'labels': labels})
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)

print("✅ HuggingFace Dataset hazır:")
print(hf_dataset)

In [ ]:
# HÜCRE 12: XLM-RoBERTa Tokenizer
print(f"📥 {MODEL_NAME} tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

print("Tokenization başlıyor (~2-4 dk sürebilir)...")
tokenized = hf_dataset.map(tokenize_fn, batched=True)
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
print("✅ Tokenization tamamlandı!")

In [ ]:
# HÜCRE 13: XLM-RoBERTa Multi-Label Modeli
print(f"🤖 {MODEL_NAME} yükleniyor...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLS),
    problem_type='multi_label_classification'
)
param_count = sum(p.numel() for p in model.parameters())
print(f"✅ Model yüklendi: {param_count:,} parametre ({param_count/1e6:.0f}M)")
print("   (UNEXPECTED/MISSING uyarıları normal — task değişikliğinden kaynaklı, sorun yok)")

---
## 🔥 BÖLÜM 4 — Eğitim

In [ ]:
# HÜCRE 14: Metrik Fonksiyonu
def compute_metrics(p: EvalPrediction):
    preds  = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    probs  = torch.sigmoid(torch.tensor(preds)).numpy()
    y_pred = (probs > 0.5).astype(int)
    y_true = p.label_ids

    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)

    try:
        roc_auc = roc_auc_score(y_true, probs, average='macro', multi_class='ovr')
    except ValueError:
        roc_auc = 0.0

    return {'f1_macro': f1_macro, 'f1_micro': f1_micro, 'roc_auc': roc_auc}

print("✅ Metrik fonksiyonu tanımlandı.")

In [ ]:
# HÜCRE 15: Trainer Konfigürasyonu
V5_CHECKPOINT_DIR = os.path.join(MODELS_DIR, 'toxicguard_v5_1_checkpoints')

training_args = TrainingArguments(
    output_dir=V5_CHECKPOINT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=200,
    warmup_steps=500,            # warmup_ratio deprecated olduğu için warmup_steps kullan
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
)

print("✅ Trainer hazır!")
print(f"   Eğitim seti: {len(tokenized['train']):,} örnek")
print(f"   Test seti  : {len(tokenized['test']):,} örnek")

In [ ]:
# HÜCRE 16: 🚀 EĞİTİMİ BAŞLAT!
print("🚀 ToxicGuard V5.1 Eğitimi Başlıyor!")
print("-" * 55)
trainer.train()
print("\n✅ V5.1 Eğitimi Tamamlandı!")

---
## ⚙️ BÖLÜM 5 — Etiket Bazlı Threshold Optimizasyonu

In [ ]:
# HÜCRE 17: Threshold Optimizasyonu
print("🔧 Her etiket için optimal threshold hesaplanıyor...")

val_output = trainer.predict(tokenized['test'])
raw_logits = val_output.predictions[0] if isinstance(val_output.predictions, tuple) else val_output.predictions
val_probs  = torch.sigmoid(torch.tensor(raw_logits)).numpy()
val_labels = val_output.label_ids

thresholds = {}
print(f"\n{'Etiket':<18} {'Opt. Threshold':>14} {'F1@0.5':>8} {'F1@opt':>8}")
print("-" * 52)

for i, label in enumerate(LABEL_COLS):
    y_true  = val_labels[:, i]
    probs_i = val_probs[:, i]

    f1_at_default = f1_score(y_true, (probs_i > 0.5).astype(int), zero_division=0)

    best_t  = 0.5
    best_f1 = 0.0
    for t in np.arange(0.10, 0.91, 0.05):
        f1 = f1_score(y_true, (probs_i > t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t  = t

    thresholds[label] = round(float(best_t), 2)
    print(f"  {label:<18} {best_t:>14.2f} {f1_at_default:>8.4f} {best_f1:>8.4f}")

print("-" * 52)
print(f"\n✅ Optimize Threshold'lar: {thresholds}")

In [ ]:
# HÜCRE 18: Final Değerlendirme
y_pred_opt = np.zeros_like(val_probs, dtype=int)
for i, label in enumerate(LABEL_COLS):
    y_pred_opt[:, i] = (val_probs[:, i] > thresholds[label]).astype(int)

y_pred_def = (val_probs > 0.5).astype(int)

f1_def = f1_score(val_labels, y_pred_def, average='macro', zero_division=0)
f1_opt = f1_score(val_labels, y_pred_opt, average='macro', zero_division=0)

try:
    roc_auc = roc_auc_score(val_labels, val_probs, average='macro', multi_class='ovr')
except:
    roc_auc = 0.0

print("📊 Final Değerlendirme:")
print("=" * 55)
print(f"  F1-macro (threshold=0.5)  : {f1_def:.4f}")
print(f"  F1-macro (opt. threshold) : {f1_opt:.4f}  {'✅ İyileşti!' if f1_opt > f1_def else ''}")
print(f"  ROC-AUC                   : {roc_auc:.4f}")
print("=" * 55)

V5_BASELINE = 0.6967
V3_BASELINE = 0.693
if f1_opt > V5_BASELINE:
    print(f"\n🎉 V5 baseline ({V5_BASELINE}) AŞILDI! → {f1_opt:.4f}")
elif f1_opt > V3_BASELINE:
    print(f"\n✅ V3 baseline aşıldı ({f1_opt:.4f}), V5 baseline'a yakın.")
else:
    print(f"\n⚠️  Henüz V3 baseline'ın altında. ({f1_opt:.4f})")

---
## 🧪 BÖLÜM 6 — Sarkazm & Kenar Durum Testleri

**V5'ten bilinen sorunlar:**
- `'You're killing it, congrats!'` → yanlışlıkla toksik diyordu
- `'Oh harika fikir...'` (TR sarkazm) → kaçırıyordu

Bu veriler düzeldi mi?

In [ ]:
# HÜCRE 19: Sarkazm & Kenar Durum Testleri
test_cases = [
    # (metin, beklenen, v5_sonucu_referans)
    ("Oh great idea, killing people is a perfect solution!",  "🔴 Toksik",   "V5: toxic 0.81 ✅"),
    ("You're absolutely killing it, congrats!",               "🟢 Güvenli",  "V5: toxic 0.53 ❌ FALSE POS"),
    ("Wow, what a brilliant move genius 🙄",                  "🟡 Sarkastik","V5: toxic 0.39 ✅"),
    ("I hate you so much!",                                   "🔴 Toksik",   "V5: toxic 0.81 ✅"),
    ("This is the best movie I've ever seen!",                "🟢 Güvenli",  "V5: toxic 0.53 ❌ FALSE POS"),
    ("Oh harika fikir, insanları öldürmek tam çözüm",         "🔴 Toksik TR","V5: toxic 0.07 ❌ KAÇIRILDI"),
    ("Çok iyi iş çıkardın, bravo!",                          "🟢 Güvenli",  "V5: toxic 0.00 ✅"),
    ("Tabii ya, sen her şeyi biliyorsun değil mi 😒",         "🟡 Sarkastik","V5: toxic 0.37 ✅"),
    ("Go kill yourself, nobody likes you",                    "🔴 Tehdit",   "V5: toxic 0.92 ✅"),
    ("That presentation was... interesting.",                 "🟡 Pasif-Agr","V5: toxic 0.01 ✅"),
]

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print("🧪 SARKAZM & KENAR DURUM TESTLERİ — V5.1 vs V5 Karşılaştırması")
print("=" * 75)

for text, expected, v5_ref in test_cases:
    inputs = tokenizer(
        text, return_tensors='pt', truncation=True, max_length=128, padding=True
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits).cpu().numpy()[0]

    detected = []
    for i, label in enumerate(LABEL_COLS):
        if probs[i] > thresholds[label]:
            detected.append(f"{label}({probs[i]:.2f})")

    toxic_score = probs[0]
    model_says = "🔴" if toxic_score > thresholds['toxic'] else "🟢"
    correct = "✅" if (model_says == "🔴") == ("🔴" in expected) else "❌"

    print(f"\n  {correct} {model_says} Metin   : {text[:60]}")
    print(f"       Beklenen: {expected}")
    print(f"       V5.1    : {', '.join(detected) if detected else 'TEMİZ'} (toxic={toxic_score:.3f})")
    print(f"       V5 Ref  : {v5_ref}")

---
## 💾 BÖLÜM 7 — V5.1 Modelini Kaydet

In [ ]:
# HÜCRE 20: V5.1 Modeli, Tokenizer ve Threshold Kaydetme
V5_FINAL_DIR = os.path.join(MODELS_DIR, 'toxicguard_v5_1_sarcasm')
os.makedirs(V5_FINAL_DIR, exist_ok=True)

trainer.save_model(V5_FINAL_DIR)
tokenizer.save_pretrained(V5_FINAL_DIR)

threshold_path = os.path.join(V5_FINAL_DIR, 'v5_thresholds.json')
with open(threshold_path, 'w') as f:
    json.dump(thresholds, f, indent=2)

results_summary = {
    'model': 'ToxicGuard V5.1',
    'base_model': MODEL_NAME,
    'total_training_samples': len(tokenized['train']),
    'total_test_samples': len(tokenized['test']),
    'f1_macro_default_threshold': round(float(f1_def), 4),
    'f1_macro_optimized_threshold': round(float(f1_opt), 4),
    'roc_auc': round(float(roc_auc), 4),
    'thresholds': thresholds,
    'label_cols': LABEL_COLS,
    'datasets_used': [
        'Kaggle Jigsaw (original)',
        'Jigsaw Unintended Bias (if available)',
        'SemEval-2018 Irony',
        'SARC Reddit Corpus (if available)',
        'Overfit-GM Turkish',
        'Toygar Turkish Offensive (replaces OffensEval-TR)'
    ]
}

results_path = os.path.join(RESULTS_DIR, 'v5_1_results_summary.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, ensure_ascii=False, indent=2)

print(f"🎉 V5.1 başarıyla kaydedildi!")
print(f"   Model       : {V5_FINAL_DIR}")
print(f"   Thresholds  : {threshold_path}")
print(f"   Sonuç özeti : {results_path}")
print(f"\n📊 FINAL SONUÇLAR:")
print(f"   F1-macro  : {f1_opt:.4f}")
print(f"   ROC-AUC   : {roc_auc:.4f}")

---
## 📈 BÖLÜM 8 — Versiyon Karşılaştırma Tablosu

In [ ]:
# HÜCRE 21: Karşılaştırma Tablosu
comparison = pd.DataFrame([
    {'Versiyon': 'V1 XGBoost',       'F1-macro': 0.599,  'ROC-AUC': 0.967,
     'Veri': 'Kaggle EN',            'Sarkazm': '❌', 'Türkçe': '❌'},
    {'Versiyon': 'V2 SVM opt.',      'F1-macro': 0.599,  'ROC-AUC': 0.967,
     'Veri': 'Kaggle EN',            'Sarkazm': '❌', 'Türkçe': '❌'},
    {'Versiyon': 'V3 DistilBERT',    'F1-macro': 0.693,  'ROC-AUC': 0.978,
     'Veri': 'Kaggle EN dengeli',    'Sarkazm': '❌', 'Türkçe': '❌'},
    {'Versiyon': 'V4 XLM-RoBERTa',  'F1-macro': '—',    'ROC-AUC': '—',
     'Veri': 'EN+TR (~50K)',         'Sarkazm': '⚠️',  'Türkçe': '⚠️'},
    {'Versiyon': 'V5 XLM-RoBERTa',  'F1-macro': 0.6967, 'ROC-AUC': 0.9818,
     'Veri': 'EN+TR+Sarkazm(130K)',  'Sarkazm': '⚠️',  'Türkçe': '✅'},
    {'Versiyon': 'V5.1 (Bu model)',
     'F1-macro': round(float(f1_opt), 4), 'ROC-AUC': round(float(roc_auc), 4),
     'Veri': f'EN+TR+Sarkazm(~{len(df_mixed)//1000}K)',
     'Sarkazm': '✅', 'Türkçe': '✅'},
])

print("\n📊 ToxicGuard Versiyon Karşılaştırması:")
print(comparison.to_string(index=False))

comparison_path = os.path.join(RESULTS_DIR, 'version_comparison_v5_1.csv')
comparison.to_csv(comparison_path, index=False)
print(f"\n✅ Kaydedildi: {comparison_path}")

---
## 🚀 Sonraki Adımlar

### Eksik Kalan Veri Setleri (Eklenirse Daha Güçlü Olur)

**Jigsaw Unintended Bias** (~50K ek satır):
1. https://kaggle.com/c/jigsaw-unintended-bias-in-toxicity-classification
2. Data → train.csv → indir → ismini `jigsaw_bias_train.csv` yap
3. **Google Drive → MyDrive → ToxicGuard → data** klasörüne sürükle

**SARC Reddit** (~30K sarkastik yorum):
1. https://kaggle.com/datasets/danofer/sarcasm
2. `train-balanced-sarcasm.csv` → indir
3. **Google Drive → MyDrive → ToxicGuard → data** klasörüne sürükle

Sonra bu notebook'u tekrar çalıştır — otomatik eklenecek!

### Streamlit'te V5.1 Modelini Kullanmak İçin:
```python
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = 'ToxicGuard/models/toxicguard_v5_1_sarcasm'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_DIR)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

with open(f'{MODEL_DIR}/v5_thresholds.json') as f:
    thresholds = json.load(f)
```
